In [ ]:
# Célula 0: Instalação e Bootstrap de Dependências (Padrão Oficial FGV)
%pip install -q numpy pandas matplotlib seaborn scikit-learn
print("✅ Dependências verificadas com sucesso!")


# 🏦 Torre de Controle de Campanhas & Funis Santander
### Framework de Analytics de Jornadas Digitais, Atribuição Causal e Previsão com Redes Neurais
**Autora:** Talita Fonseca  
**Instituição:** Fundação Getulio Vargas (FGV) — MBA em Inteligência Artificial & Analytics  
**Professor Responsável:** Prof. Marcelo Fidos Jr.  
**Aplicação Online:** [https://atalitafonseca.github.io/](https://atalitafonseca.github.io/)  
**Repositório Oficial:** [github.com/atalitafonseca/atalitafonseca.github.io](https://github.com/atalitafonseca/atalitafonseca.github.io)

---

## 📌 1. Contexto de Negócio & Diagnóstico dos Silos

Com mais de **19 milhões de clientes ativos diários**, o aplicativo do Santander movimenta centenas de milhões de eventos por dia. Contudo, as operações enfrentam três grandes gargalos:
1. **Desconexão da Tríade de Negócio:** CRM, Produto e Financeiro operam em silos analíticos sem conexão ponta a ponta.
2. **Silo de Acesso aos Atributos de Clientes:** Apenas o time de CRM acessa as tabelas ricas de clientes (`nrpess`). O time de Produto, que desenha as jornadas e ofertas, depende de solicitações manuais lentas.
3. **Superatribuição de 10 Dias e Inviabilidade de Grupos de Controle:** A regra legada de 10 dias do CRM gera falsos positivos ao creditar transações orgânicas como mérito de marketing. Além disso, travar clientes em grupos de controle é inviável por gerar perda imediata de faturamento comercial.

### 🎯 Objetivo do Projeto
Construir uma **Camada Semântica Padronizada (Gold/Platinum)** e um **Modelo Preditivo com Redes Neurais (MLP)** que:
* Realiza **Atribuição Causal sem Grupo de Controle** descontando a probabilidade orgânica base do cliente.
* Permite ao time de Produto explorar o Dicionário de Atributos, calcular o **Sizing de Audiência** e prever a conversão por espaço comercial (*Lightbox, Alert, Banner, Push, Email*).
* Fornece uma **Torre de Ritmo (Pacing MTD)** com comparativo do mês anterior e **Forecast Preditivo de Fechamento do Mês**.


In [ ]:
# Configuração do Ambiente e Bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Modelagem & Redes Neurais
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
np.random.seed(42)

print("✅ Bibliotecas carregadas com sucesso!")


---
## 🏗️ 2. Engenharia de Dados: Ingestão e Unificação das 3 Tabelas via `nrpess`

Simulamos o ecossistema corporativo do Santander integrando:
1. **Dicionário de Atributos de Clientes (`silver_atributos_clientes`):** Perfil (*Especial, Select, Private*), Score ARPAC de Rentabilidade, Outflow/Evasão para fintechs, Consentimento Open Finance, Salário/FOPA e Hábitos Transacionais.
2. **Base de Campanhas & Espaços Comerciais (`silver_campanhas_crm`):** Espaço de veiculação (*Lightbox, Alert, Banner, Push, Email*), timestamps de visualização e clique por `nrpess`.
3. **Base de Clickstream & Produção (`silver_jornadas_producao`):** 7 etapas de funil (Visualização, Clique, Entrada App, Simulação, ID Santander, Fechamento App) e liquidação efetiva no Core Bancário.


In [ ]:
# 1. Base de Atributos de Clientes (Feature Store de CRM)
n_clientes = 25000
np.random.seed(42)

nrpess_list = [f"nrpess_{i:06d}" for i in range(n_clientes)]
segmentos = np.random.choice(['Especial', 'Select', 'Private'], size=n_clientes, p=[0.60, 0.32, 0.08])
score_arpac = np.round(np.random.beta(5, 2, size=n_clientes) * 10, 1) # Score ARPAC de 0 a 10
open_finance = np.random.choice(['Não Possui', 'Possui Ativo'], size=n_clientes, p=[0.62, 0.38])
cliente_fopa = np.random.choice(['Sim (É Folha)', 'Não (Sem Folha)'], size=n_clientes, p=[0.48, 0.52])

# Hábitos e gastos condicionados ao segmento
gasto_cartao = np.where(segmentos == 'Private', np.random.normal(9500, 2200, n_clientes),
               np.where(segmentos == 'Select', np.random.normal(4800, 1200, n_clientes),
                        np.random.normal(1900, 600, n_clientes))).clip(min=100)

freq_pix = np.where(segmentos == 'Private', np.random.poisson(24, n_clientes),
           np.where(segmentos == 'Select', np.random.poisson(14, n_clientes),
                    np.random.poisson(8, n_clientes))).clip(min=0)

freq_boletos = np.where(segmentos == 'Private', np.random.poisson(8, n_clientes),
               np.where(segmentos == 'Select', np.random.poisson(5, n_clientes),
                        np.random.poisson(2, n_clientes))).clip(min=0)

prob_organica_base = (freq_pix * 0.025 + freq_boletos * 0.04 + (gasto_cartao / 12000.0) * 0.25).clip(0.05, 0.95)

df_atributos = pd.DataFrame({
    'nrpess': nrpess_list,
    'segmento': segmentos,
    'score_arpac': score_arpac,
    'open_finance': open_finance,
    'cliente_fopa': cliente_fopa,
    'gasto_cartao_mes': np.round(gasto_cartao, 2),
    'freq_pix_mes': freq_pix,
    'freq_boletos_mes': freq_boletos,
    'prob_organica_base': np.round(prob_organica_base, 3)
})

# 2. Base de Campanhas e Espaços Comerciais
n_campanhas = 35000
users_camp = np.random.choice(nrpess_list, size=n_campanhas)
espacos = ['Lightbox', 'Alert', 'Banner', 'Push', 'Email']
produtos_camp = ['Pix', 'Boleto', 'Pix Automático', 'Pix Parcelado', 'Upgrade', 'Cartão de Crédito', 'Limite da Conta']

base_dt = datetime(2026, 8, 1, 8, 0, 0)
dt_exposicao = [base_dt + timedelta(days=float(np.random.uniform(0, 20)), hours=float(np.random.uniform(0, 24))) for _ in range(n_campanhas)]
cliques = np.random.choice([1, 0], size=n_campanhas, p=[0.18, 0.82])

df_campanhas = pd.DataFrame({
    'id_campanha': [f"CAMP_{i:04d}" for i in np.random.randint(100, 115, size=n_campanhas)],
    'nrpess': users_camp,
    'nome_do_produto': np.random.choice(produtos_camp, size=n_campanhas, p=[0.25, 0.20, 0.15, 0.12, 0.10, 0.10, 0.08]),
    'espaco_veiculacao': np.random.choice(espacos, size=n_campanhas, p=[0.35, 0.25, 0.20, 0.12, 0.08]),
    'dt_exposicao': dt_exposicao,
    'clicou_flag': cliques
})

# 3. Base de Clickstream e Produção
n_sessoes = 40000
users_sess = np.random.choice(nrpess_list, size=n_sessoes)
dt_sess = [base_dt + timedelta(days=float(np.random.uniform(0, 22)), hours=float(np.random.uniform(0, 24))) for _ in range(n_sessoes)]

df_jornadas = pd.DataFrame({
    'session_id': [f"sess_{i:06d}" for i in range(n_sessoes)],
    'nrpess': users_sess,
    'dt_jornada': dt_sess,
    'tempo_tela_segundos': np.random.normal(48, 15, size=n_sessoes).clip(min=5)
})

print(f"📊 Base Atributos CRM: {df_atributos.shape[0]:,} clientes")
print(f"📊 Base Campanhas:     {df_campanhas.shape[0]:,} interações")
print(f"📊 Base Clickstream:   {df_jornadas.shape[0]:,} sessões de app")


---
## 🔗 3. Atribuição Causal sem Grupo de Controle (Desconto da Probabilidade Orgânica)

Implementamos a fórmula de Atribuição Causal Dinâmica:
$$w_{\text{causal}} = e^{-\lambda \Delta t} \times (1 - P_{\text{orgânica}})$$
Onde $\Delta t$ é o tempo entre a visualização da campanha e a transação (com meia-vida $\lambda = 12h$).


In [ ]:
# Unificação por nrpess (Criação da Camada Gold)
df_gold = pd.merge(df_jornadas, df_atributos, on='nrpess', how='inner')
df_gold = pd.merge(df_gold, df_campanhas, on='nrpess', how='left')

# Filtrar causalidade temporal (campanha antes da sessão)
df_gold = df_gold[(df_gold['dt_exposicao'].isna()) | (df_gold['dt_exposicao'] <= df_gold['dt_jornada'])].copy()
df_gold['delta_horas'] = (df_gold['dt_jornada'] - df_gold['dt_exposicao']).dt.total_seconds() / 3600.0

# Manter a campanha mais recente anterior à jornada
df_gold = df_gold.sort_values(['session_id', 'dt_exposicao'], ascending=[True, False]).drop_duplicates(subset=['session_id']).copy()

# Preenchimento de tráfego orgânico
df_gold['espaco_veiculacao'] = df_gold['espaco_veiculacao'].fillna('Orgânico Puro (Sem Campanha)')
df_gold['nome_do_produto'] = df_gold['nome_do_produto'].fillna('Transação Padrão')
df_gold['delta_horas'] = df_gold['delta_horas'].fillna(999.0)

# Cálculo da Atribuição Causal sem Grupo Controle
lambda_decay = np.log(2) / 12.0
peso_tempo = np.where(df_gold['delta_horas'] <= 72, np.exp(-lambda_decay * df_gold['delta_horas']), 0.0)
df_gold['peso_atribuicao_causal'] = peso_tempo * (1.0 - df_gold['prob_organica_base'])

# Geração do Status de Produção (Venda/Contratação Concluída no Core)
prob_conversao_real = (
    0.28 * (df_gold['score_arpac'] >= 7.0).astype(int) +
    0.22 * (df_gold['espaco_veiculacao'] == 'Lightbox').astype(int) +
    0.14 * (df_gold['espaco_veiculacao'] == 'Alert').astype(int) +
    0.12 * (df_gold['cliente_fopa'] == 'Sim (É Folha)').astype(int) +
    0.10 * (df_gold['clicou_flag'].fillna(0)) +
    0.14 * df_gold['peso_atribuicao_causal']
).clip(0.05, 0.95)

df_gold['venda_concluida_flag'] = (np.random.uniform(0, 1, size=len(df_gold)) < prob_conversao_real).astype(int)

print("🌟 Camada Gold Unificada gerada com sucesso!")
print(f"Total de registros na Gold: {df_gold.shape[0]:,} sessões unificadas.")


---
## 🧠 4. Modelagem Preditiva: Baseline (Regressão Logística) vs Rede Neural (MLPClassifier)

Conforme exigência do curso da FGV, comparamos o modelo baseline linear contra uma **Rede Neural Densa Multi-Layer Perceptron (MLP)** de 2 camadas ocultas.


In [ ]:
# Preparação da Matriz de Features
features_num = ['score_arpac', 'gasto_cartao_mes', 'freq_pix_mes', 'freq_boletos_mes', 'tempo_tela_segundos', 'prob_organica_base', 'peso_atribuicao_causal']
features_cat = ['segmento', 'open_finance', 'cliente_fopa', 'espaco_veiculacao', 'nome_do_produto']

X = df_gold[features_num + features_cat].copy()
y = df_gold['venda_concluida_flag'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), features_cat)
    ]
)

# 1. Baseline: Regressão Logística
pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

# 2. Modelo Campeão: Rede Neural Densa (MLPClassifier)
pipe_mlp = Pipeline([
    ('prep', preprocessor),
    ('clf', MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=120,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=42
    ))
])

# Treinamento
pipe_lr.fit(X_train, y_train)
pipe_mlp.fit(X_train, y_train)

# Predições
y_pred_lr = pipe_lr.predict(X_test)
y_prob_lr = pipe_lr.predict_proba(X_test)[:, 1]

y_pred_mlp = pipe_mlp.predict(X_test)
y_prob_mlp = pipe_mlp.predict_proba(X_test)[:, 1]

# Comparativo de Métricas
df_metricas = pd.DataFrame({
    'Métrica': ['Acurácia', 'Precisão', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Regressão Logística (Baseline)': [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
        roc_auc_score(y_test, y_prob_lr)
    ],
    'Rede Neural MLP (Campeão)': [
        accuracy_score(y_test, y_pred_mlp),
        precision_score(y_test, y_pred_mlp),
        recall_score(y_test, y_pred_mlp),
        f1_score(y_test, y_pred_mlp),
        roc_auc_score(y_test, y_prob_mlp)
    ]
})

print("🏆 Tabela Comparativa de Performance:")
display(df_metricas.round(4))


In [ ]:
# Curva ROC e Curva de Perda (Loss Curve) da Rede Neural
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

# 1. Curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, y_prob_mlp)

ax1.plot(fpr_mlp, tpr_mlp, color='#ec0000', lw=2.5, label=f"Rede Neural MLP (AUC = {roc_auc_score(y_test, y_prob_mlp):.3f})")
ax1.plot(fpr_lr, tpr_lr, color='#0284c7', lw=2, linestyle='--', label=f"Regressão Logística (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})")
ax1.plot([0, 1], [0, 1], color='gray', linestyle=':')
ax1.set_title("Curva ROC: Baseline vs Rede Neural MLP (FGV)")
ax1.set_xlabel("Taxa de Falsos Positivos")
ax1.set_ylabel("Taxa de Verdadeiros Positivos (Recall)")
ax1.legend(loc='lower right')

# 2. Curva de Perda (Loss Curve por Época)
mlp_loss = pipe_mlp.named_steps['clf'].loss_curve_
ax2.plot(range(1, len(mlp_loss) + 1), mlp_loss, color='#ec0000', lw=2.5, marker='.')
ax2.set_title("Convergência da Rede Neural (Binary Cross-Entropy Loss)")
ax2.set_xlabel("Épocas de Treinamento")
ax2.set_ylabel("Loss")

plt.tight_layout()
plt.show()


---
## 🎲 5. Validação de Robustez Estocástica Multi-Seed (Padrão FGV: Seeds 42, 7, 123)

Executamos o treinamento em **3 sementes aleatórias distintas** para comprovar estabilidade estatística.


In [ ]:
# Loop Multi-Seed Oficial da FGV
seeds = [42, 7, 123]
results_lr = []
results_mlp = []

for s in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=s, stratify=y)
    
    # Baseline
    m_lr = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=s))])
    m_lr.fit(X_tr, y_tr)
    prob_l = m_lr.predict_proba(X_te)[:, 1]
    pred_l = m_lr.predict(X_te)
    results_lr.append({
        'seed': s, 'auc': roc_auc_score(y_te, prob_l), 'f1': f1_score(y_te, pred_l), 'rec': recall_score(y_te, pred_l)
    })
    
    # Rede Neural
    m_mlp = Pipeline([('prep', preprocessor), ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=120, random_state=s))])
    m_mlp.fit(X_tr, y_tr)
    prob_m = m_mlp.predict_proba(X_te)[:, 1]
    pred_m = m_mlp.predict(X_te)
    results_mlp.append({
        'seed': s, 'auc': roc_auc_score(y_te, prob_m), 'f1': f1_score(y_te, pred_m), 'rec': recall_score(y_te, pred_m)
    })

df_res_lr = pd.DataFrame(results_lr)
df_res_mlp = pd.DataFrame(results_mlp)

print("📊 Resultados Finais Multi-Seed (Média ± Desvio-Padrão):")
print(f"• Baseline LogReg: AUC = {df_res_lr['auc'].mean():.4f} ± {df_res_lr['auc'].std():.4f} | F1 = {df_res_lr['f1'].mean():.4f} ± {df_res_lr['f1'].std():.4f}")
print(f"• Rede Neural MLP: AUC = {df_res_mlp['auc'].mean():.4f} ± {df_res_mlp['auc'].std():.4f} | F1 = {df_res_mlp['f1'].mean():.4f} ± {df_res_mlp['f1'].std():.4f}")


---
## 🎯 6. Conector de Simulação de Negócio: Conversão vs Awareness

Demonstração do motor de simulação que alimenta a **Aba 1 (Simulador)** e a **Aba 3 (Torre de Pacing)** do Dashboard web.


In [ ]:
# Função de Simulação de Audiência & Conversão / Awareness
def simular_campanha(produto='Pix', espaco='Lightbox', segmentos=['Especial', 'Select', 'Private'], open_finance_ativo=True, arpac_min=7.0, modo='conversao'):
    base_universo = 8500000
    
    # Filtro de segmento
    seg_mult = sum([0.60 if s == 'Especial' else 0.32 if s == 'Select' else 0.08 for s in segmentos])
    
    # Alavancas de IA
    of_mult = 1.0 if open_finance_ativo else 0.72  # Trava de Open Finance reduz público em 28%
    arpac_mult = 0.62 if arpac_min >= 7.0 else 0.95
    
    alcance_publico = int(base_universo * seg_mult * of_mult * arpac_mult)
    
    view_rates = {'Lightbox': 0.70, 'Alert': 0.82, 'Banner': 0.50, 'Push': 0.38, 'Email': 0.25}
    ctrs = {'Lightbox': 0.18, 'Alert': 0.14, 'Banner': 0.08, 'Push': 0.06, 'Email': 0.035}
    
    vr = view_rates.get(espaco, 0.70)
    ctr = ctrs.get(espaco, 0.18)
    
    views = int(alcance_publico * vr)
    cliques = int(views * ctr)
    contratos_finais = int(cliques * 0.80 * 0.82 * 0.742)
    
    if modo == 'awareness':
        return {
            'Modo': 'Awareness & Alcance de Marca',
            'Público Elegível': f"{alcance_publico:,}",
            'Pessoas Alcançadas': f"{views:,}",
            'Cliques & Engajamento': f"{cliques:,}",
            'Frequência Média': '2.2x',
            'CPM Estimado': 'R$ 4,80'
        }
    else:
        return {
            'Modo': 'Conversão & Vendas',
            'Público Elegível': f"{alcance_publico:,}",
            'Views no Espaço': f"{views:,}",
            'Contratos Finais Core': f"{contratos_finais:,}",
            'Taxa de Conversão': f"{(contratos_finais/views)*100:.2f}%"
        }

# Teste de Simulação
sim_conv = simular_campanha(produto='Pix', espaco='Lightbox', open_finance_ativo=True, modo='conversao')
sim_aware = simular_campanha(produto='Pix', espaco='Lightbox', open_finance_ativo=True, modo='awareness')

print("🎯 Simulação no Modo Conversão & Vendas:")
print(sim_conv)
print("
📢 Simulação no Modo Awareness & Alcance:")
print(sim_aware)


---
## 📑 7. Anexo: Outras Aplicações de Negócio com a Mesma Técnica

1. **Prevenção Inteligente de Churn (Evasão em Contas e Cartões):** Identificação de perda de engajamento do cliente e acionamento de benefícios no momento ideal.
2. **Recomendação de Produtos de Investimentos (Next-Best-Asset):** Oferta de ativos de liquidez no momento pós-resgate de Pix ou no extrato.
3. **Detecção de Fricções de UX em Tempo Real:** Identificação de hesitação no app para acionamento de rotas de suporte antes da desistência do usuário.
